## But

Vérifier la validation du pipeline prévu dans AR1

## Principe général

* Historique complet **1959–2025** sur **UNRATE**.
* Données consommées exclusivement via **Feast**.
* Modèle Baseline et univarié.
* Backtesting temporel minimal avec prévisions ponctuelles et intervalles de prédiction conformes (95 %).
* Comparer AR1 et ARp

## Résultat
cf la fin

# Package

In [18]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# ----------------------------
# Nixtla
# ----------------------------
from statsforecast import StatsForecast
from statsforecast.models import AutoRegressive
from statsforecast.utils import ConformalIntervals
from utilsforecast.plotting import plot_series

# Importation des données

In [ ]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()


# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Dictionnaire de modèle

In [20]:
from statsforecast.models import AutoRegressive

SF_MODELS = {
    "AR_1":   lambda: AutoRegressive(lags=1)}

# Backtesting

In [21]:
from mlforecast.utils import PredictionIntervals

def run_backtesting_h12_simple(
    mlf,
    ts,
    *,
    h=12,
    step_size=12,
    partitions=4,
    pi_windows=3,
    levels=[95],
):
    """
    Simple backtesting:
    - horizon h (default: 12 months)
    - step_size between cutoffs (default: 12 months)
    - few partitions (default: 4)
    - conformal prediction intervals
    """

    pi = PredictionIntervals(
        h=h,
        n_windows=pi_windows,
        method="conformal_distribution",
    )

    bkt_df = mlf.cross_validation(
        df=ts,
        h=h,
        step_size=step_size,
        n_windows=partitions,
        prediction_intervals=pi,
        level=levels,
        fitted=True,
    )

    return bkt_df

# Run 

In [31]:
# ============================================================
# (ADD-0) Fix project root (notebook in 3_notebook/)
# ============================================================
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent  # .../Explainable_AI_... (racine)
# Optionnel: os.chdir(PROJECT_ROOT)  # inutile si on utilise PROJECT_ROOT partout
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())


# ----------------------------
# Config
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"

FEATURE_REFS = ["stationary_value:value"]  # y uniquement

H = 12
STEP_SIZE = 12
PARTITIONS = 35
PI_WINDOWS = 3
LEVELS = [95]

AR_LAGS = 1  # AR(p)

# ----------------------------
# 1) entity_df
# ----------------------------
dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

# ----------------------------
# 2) Feast -> ts (format StatsForecast)
# ----------------------------
ts_raw = load_features_from_feast(entity_df=entity_df, feature_refs=FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

# ----------------------------
# 3) StatsForecast
# ----------------------------
sf = StatsForecast(
    models=[AutoRegressive(lags=AR_LAGS)],
    freq=FREQ,
)

# ----------------------------
# 4) Cross-validation + Conformal intervals
# ----------------------------
ci = ConformalIntervals(h=H, n_windows=PI_WINDOWS)

bkt_df = sf.cross_validation(
    df=ts,
    h=H,
    step_size=STEP_SIZE,
    n_windows=PARTITIONS,
    prediction_intervals=ci,
    level=LEVELS,
)

# ----------------------------
# 5) Reusable output table
# ----------------------------
# NB: le nom de la colonne modèle dépend du "alias" StatsForecast.
# Par défaut c'est souvent "AutoRegressive" (ou un nom proche).
model_col = [c for c in bkt_df.columns if c.lower().startswith("autoregressive")][0]

lo_col = [c for c in bkt_df.columns if c.lower().endswith("lo-95") and c.lower().startswith("autoregressive")][0]
hi_col = [c for c in bkt_df.columns if c.lower().endswith("hi-95") and c.lower().startswith("autoregressive")][0]

df_ar_forecasts = (
    bkt_df[["unique_id", "ds", "cutoff", "y", model_col, lo_col, hi_col]]
    .rename(columns={
        "unique_id": "series_id",
        "ds": "date",
        "y": "y_obs",
        model_col: "y_hat_ar",
        lo_col: "y_hat_ar_lo_95",
        hi_col: "y_hat_ar_hi_95",
    })
    .sort_values(["series_id", "date"])
    .reset_index(drop=True)
)

df_ar_forecasts

PROJECT_ROOT = D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


,series_id,date,cutoff,y_obs,y_hat_ar,y_hat_ar_lo_95,y_hat_ar_hi_95
0,UNRATE,1990-10-01 00:00:00+00:00,1990-09-01 00:00:00+00:00,0.6,0.576313,0.466979,0.685647
1,UNRATE,1990-11-01 00:00:00+00:00,1990-09-01 00:00:00+00:00,0.8,0.553541,0.371908,0.735175
2,UNRATE,1990-12-01 00:00:00+00:00,1990-09-01 00:00:00+00:00,0.9,0.531650,0.349087,0.714213
3,UNRATE,1991-01-01 00:00:00+00:00,1990-09-01 00:00:00+00:00,1.0,0.510606,0.376869,0.644344
4,UNRATE,1991-02-01 00:00:00+00:00,1990-09-01 00:00:00+00:00,1.3,0.490375,0.312230,0.668521
...,...,...,...,...,...,...,...
415,UNRATE,2025-05-01 00:00:00+00:00,2024-09-01 00:00:00+00:00,0.2,0.110026,-0.734424,0.954475
416,UNRATE,2025-06-01 00:00:00+00:00,2024-09-01 00:00:00+00:00,0.0,0.095865,-0.945689,1.137419
417,UNRATE,2025-07-01 00:00:00+00:00,2024-09-01 00:00:00+00:00,0.0,0.083155,-0.729203,0.895514
418,UNRATE,2025-08-01 00:00:00+00:00,2024-09-01 00:00:00+00:00,0.1,0.071748,-0.468550,0.612046


In [ ]:
# ============================================================
# (ADD) 6) Save artifacts & outputs (clean separation)
# ============================================================
from datetime import datetime
import json

OUTPUT_FORECASTS_DIR = PROJECT_ROOT / "outputs" / "forecasts"

ARTIFACT_CONFIGS_DIR = PROJECT_ROOT / "artifacts" / "configs"
ARTIFACT_CV_DIR      = PROJECT_ROOT / "artifacts" / "cv"
ARTIFACT_META_DIR    = PROJECT_ROOT / "artifacts" / "metadata"

OUTPUT_FORECASTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CV_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_META_DIR.mkdir(parents=True, exist_ok=True)

# ---- Outputs (consommation analyse)
oos_path = OUTPUT_FORECASTS_DIR / f"{SERIES_ID.lower()}_ar_lag{AR_LAGS}_oos_forecasts.parquet"
df_ar_forecasts.to_parquet(oos_path, index=False)

# ---- Artifacts (repro / debug)
bkt_path = ARTIFACT_CV_DIR / f"{SERIES_ID.lower()}_ar_lag{AR_LAGS}_bkt_raw.parquet"
bkt_df.to_parquet(bkt_path, index=False)

cfg_path = ARTIFACT_CONFIGS_DIR / f"{SERIES_ID.lower()}_ar_lag{AR_LAGS}_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

meta_path = ARTIFACT_META_DIR / f"{SERIES_ID.lower()}_ar_lag{AR_LAGS}_run_info.json"
run_info = {
    "run_utc": datetime.utcnow().isoformat(),
    "project_root": str(PROJECT_ROOT.resolve()),
    "files": {
        
        "oos": str(oos_path.resolve()),
        "bkt": str(bkt_path.resolve()),
        "config": str(cfg_path.resolve()),
    },
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)

C:\Users\Mita\AppData\Local\Temp\ipykernel_4076\874937599.py:32: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



# Graphique

In [28]:
df_obs = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })
    [["unique_id", "ds", "y"]]
)

In [29]:
df_fcst = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_hat_ar": "AR",
        "y_hat_ar_lo_95": "AR-lo-95",
        "y_hat_ar_hi_95": "AR-hi-95",
    })
    [[
        "unique_id",
        "ds",
        "AR",
        "AR-lo-95",
        "AR-hi-95",
    ]]
)

In [30]:
from utilsforecast.plotting import plot_series

fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

# Rename legend entries
for trace in fig.data:
    if trace.name == "y":
        trace.name = "Unemployment rate (%)"
    elif trace.name == "AR":
        trace.name = "AutoRegressive (AR)"
    elif "level_95" in trace.name.lower():
        trace.name = "95% Prediction Interval"

fig.show()

## Résultat
On voit notre modèle AR1 est très basique, il suit juste la direction du taux de chômage. Avec ce baseline, on s'aperçoit des informations très importantes pour orienter notre expérimentation. 

De 1990 à 2007, l'économie américaine a été stable. En 2008, elle a été frappée par la crise de Subprime. En 2019, çà été la crise de Coronavirus. Qu'est-ce qu'on peut dire de ces trois périodes? 

Globalement, le modèle auto-régressif reste proche des observations en période de stabilité. C'est une bonne capacité à capter la dynamique du chômage.

Lors des ruptures structurelles des deux crises, la qualité des prévisions se dégrade et les intervalles de prédiction s’élargissent. Ce qui traduit une incertitude de plus en plus élevée.

Le modèle capte la direction des variations, mais sa fiabilité diminue en période de crise, sans masquer cette incertitude.

Cette étude constitue un **sanity check du système de prévision**. Elle valide le comportement attendu du modèle et la cohérence du pipeline.

Une approche plus complexe est attendu. 

## Next
Essayons d'optimiser le paramètre "p" de AR pour confirmer notre analyse. 